# Driver: Questions + Profiling (Local Modules)

This notebook uses the `*_local.py` modules created in this sandbox.
In your repo, you can rename them back to the canonical names if you want.


In [1]:
from pathlib import Path

# Your DB paths
SQLITE_PATH = Path('MINIDEV/dev_databases/debit_card_specializing/debit_card_specializing.sqlite')
DESC_DIR = Path('MINIDEV/dev_databases/debit_card_specializing/database_description')

SQLITE_PATH, DESC_DIR

(PosixPath('MINIDEV/dev_databases/debit_card_specializing/debit_card_specializing.sqlite'),
 PosixPath('MINIDEV/dev_databases/debit_card_specializing/database_description'))

In [2]:
import sys
from pathlib import Path
# Ensure local modules in this project folder are importable
sys.path.insert(0, str(Path().resolve()))
print('Using project root:', Path().resolve())


Using project root: /Users/anik/Downloads/AR_DB (3)/untitled folder/t2sql


## 1) Build LONG + FULL profiles
Writes next to the sqlite file:
- `debit_card_specializing.long_profiles.jsonl`
- `debit_card_specializing.full_profiles.jsonl`


In [3]:
from profiles_sqlite_local import build_and_write_long_profiles
from profiles_full_local import build_and_write_full_profiles

long_path = build_and_write_long_profiles(
    SQLITE_PATH,
    db_id='debit_card_specializing',
    sample_n=5,
    topk=5,
    distinct_limit=None,
)
print('Wrote:', long_path)

full_path = build_and_write_full_profiles(long_path, DESC_DIR, debug=True)
print('Wrote:', full_path)


Wrote: MINIDEV/dev_databases/debit_card_specializing/debit_card_specializing.long_profiles.jsonl
[INFO] Full profile merge matched 21/21 columns with dev docs.
Wrote: MINIDEV/dev_databases/debit_card_specializing/debit_card_specializing.full_profiles.jsonl


## 1b) Build SHORT profile prompts + generate SHORT profiles (LLM)

Uses:
- `llm_backends_local.py` (Qwen + GPT-OSS backends)
- `short_profiles_local.py` (prompt creation + generation + postprocess)

Outputs are written **next to** the FULL profile JSONL:
- `<db_id>.short_profile_prompts.jsonl`
- `<db_id>.short_profiles.jsonl`


In [4]:
# from short_profiles_local import build_short_profile_prompts, generate_short_profiles
# from llm_backends_local import make_backend

# prompts_path = build_short_profile_prompts(full_path, max_profile_chars=3500)
# print("Wrote prompts:", prompts_path)

# BACKEND_KIND = "qwen"          # or "gptoss"
# MODEL_ID = None               # optional, can set explicitly
# DEVICE_MAP = "auto"
# DTYPE = None                  # if None, backend defaults to bfloat16 on GPU

# # ---- IMPORTANT: don't recreate backend on rerun ----
# if "backend" not in globals():
#     backend = make_backend(BACKEND_KIND, model_id=MODEL_ID, device_map=DEVICE_MAP, dtype=DTYPE, cache=True)
# else:
#     # if you changed kind/model in the cell, rebuild once
#     want_kind = BACKEND_KIND.lower().strip()
#     want_id = MODEL_ID or ("Qwen/Qwen2.5-7B-Instruct" if want_kind == "qwen" else "openai/gpt-oss-20b")
#     cur_id = getattr(backend, "model_id", None)
#     if cur_id != want_id:
#         backend = make_backend(BACKEND_KIND, model_id=MODEL_ID, device_map=DEVICE_MAP, dtype=DTYPE, cache=True)

# short_out = generate_short_profiles(
#     prompts_path,
#     backend=backend,
#     max_new_tokens=64,
#     overwrite=False,
#     gen_kwargs={"do_sample": False},
# )
# print("Wrote short profiles:", short_out)
from short_profiles_local import build_short_profile_prompts, generate_short_profiles
from llm_backends_local import make_backend

prompts_path = build_short_profile_prompts(full_path, max_profile_chars=3500)
print("Wrote prompts:", prompts_path)

BACKEND_KIND = "gptoss"        # ✅ CHANGED: was "qwen" (or comment). Now explicitly gptoss.
MODEL_ID = "openai/gpt-oss-20b" # ✅ CHANGED: was None. Now explicit OSS model id.
DEVICE_MAP = "auto"
DTYPE = "auto"                # ✅ CHANGED: was None. "auto" lets backend/transformers choose best dtype (MXFP4 if supported)

# ---- IMPORTANT: don't recreate backend on rerun ----
if "backend" not in globals():
    backend = make_backend(BACKEND_KIND, model_id=MODEL_ID, device_map=DEVICE_MAP, dtype=DTYPE, cache=True)  # ✅ CHANGED: passes explicit MODEL_ID + dtype="auto"
else:
    # if you changed kind/model in the cell, rebuild once
    want_kind = BACKEND_KIND.lower().strip()  # (same)
    want_id = MODEL_ID                       # ✅ CHANGED: simpler + exact (no default-switching logic needed)

    cur_kind = getattr(backend, "kind", None)        # ✅ CHANGED: also compare backend kind (prevents mismatched reuse)
    cur_id = getattr(backend, "model_id", None)      # (same idea)

    if (cur_kind != want_kind) or (cur_id != want_id):  # ✅ CHANGED: rebuild if kind OR id changed
        backend = make_backend(BACKEND_KIND, model_id=MODEL_ID, device_map=DEVICE_MAP, dtype=DTYPE, cache=True)  # ✅ CHANGED

short_out = generate_short_profiles(
    prompts_path,
    backend=backend,
    max_new_tokens=128,
    overwrite=False,
    gen_kwargs={
        "do_sample": False,
        "temperature": 0.0,   # ✅ CHANGED/ADDED: makes deterministic generation extra-stable
    },
)
print("Wrote short profiles:", short_out)


Wrote prompts: MINIDEV/dev_databases/debit_card_specializing/debit_card_specializing.short_profile_prompts.jsonl


/home/ap2645/.local/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Wrote short profiles: MINIDEV/dev_databases/debit_card_specializing/debit_card_specializing.short_profiles.jsonl


## 2) Dependency tree + literals (token-index aligned)


## 8) Offline AR-DB build (concepts + segments + PK-star edges)

This step is **question-free**: it reads the SQLite schema (and optionally your `*.short_profiles.jsonl`)
and writes a single AR-DB JSON you can later use at query time for resolver / traversal.

Outputs:
- `<db_id>.ardb_offline.json`
